In [1]:
n_jobs= 60
#libraries
import pandas as pd
import numpy as np
import os
import sys
from glob import glob
import joblib
import warnings
from datetime import date, datetime
import copy

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from scipy.stats import pearsonr
import scipy.stats as st

from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning

import pickle

from sklearn.compose import TransformedTargetRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import PowerTransformer

from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [2]:
def MLenet(X, y, X_test):
    # Define the pipeline
    pipeline = Pipeline([('scaler', StandardScaler()),  # Standardize features
                         ('model', TransformedTargetRegressor(
                             regressor=ElasticNet(),  # ElasticNet as the model
                             transformer=Pipeline([
                                 ('scaler', StandardScaler()),
                                 ('power', PowerTransformer(method='yeo-johnson')),  # Apply PowerTransformer to the target
                                 ])
                          ))
                        ])
    
    # Define hyperparameter grid for ElasticNet
    param_grid = {
        'model__regressor__alpha': np.logspace(-1, 2, 70),
        'model__regressor__l1_ratio': np.linspace(0,1,25)
    }
    
    # Set up GridSearchCV
    grid_search = GridSearchCV(pipeline,
                                param_grid,
                                cv=3,
                                scoring='r2',
                                n_jobs=n_jobs,
                                )
    
    # Fit the pipeline
    grid_search.fit(X, y)

    #predict (train)
    y_pred = grid_search.predict(X)

    #performance (train)
    fullmodel = grid_search
    model = grid_search.best_estimator_

    #predict (test)
    y_pred_test = model.predict(X_test)

    return fullmodel, model, y_pred, y_pred_test


In [3]:
def MLpls(X, y, X_test):
    # Define the pipeline
    pipeline = Pipeline([('scaler', StandardScaler()),  # Standardize features
                         ('model', TransformedTargetRegressor(
                             regressor=PLSRegression(),  # PLS as the model
                             transformer=Pipeline([
                                 ('scaler', StandardScaler()),
                                 ('power', PowerTransformer(method='yeo-johnson')),  # Apply PowerTransformer to the target
                                 ])
                          ))
                        ])
    
    # Define hyperparameter grid for ElasticNet
    param_grid = {
        'model__regressor__n_components': np.linspace(1,200,100, dtype=int),
    }
    
    # Set up GridSearchCV
    grid_search = GridSearchCV(pipeline,
                                param_grid,
                                cv=3,
                                scoring='r2',
                                n_jobs=n_jobs,
                                )
    
    # Fit the pipeline
    grid_search.fit(X, y)

    #predict (train)
    y_pred = grid_search.predict(X)

    #performance (train)
    fullmodel = grid_search
    model = grid_search.best_estimator_

    #predict (test)
    y_pred_test = model.predict(X_test)

    return fullmodel, model, y_pred, y_pred_test


In [4]:
def MLkrr(X, y, X_test):
    # Define the pipeline
    pipeline = Pipeline([('scaler', StandardScaler()),  # Standardize features
                         ('model', TransformedTargetRegressor(
                             regressor=KernelRidge(),  #  the model
                             transformer=Pipeline([
                                 ('scaler', StandardScaler()),
                                 ('power', PowerTransformer(method='yeo-johnson')),  # Apply PowerTransformer to the target
                                 ])
                          ))
                        ])
    
    # Define hyperparameter grid for ml
    param_grid = {
                    'model__regressor__alpha': np.logspace(-3, 3, 50),
                    'model__regressor__kernel': ['linear', 'rbf'],
                    'model__regressor__gamma': np.logspace(-3, 2, 10)  # only relevant for rbf
                }
    
    # Set up GridSearchCV
    grid_search = GridSearchCV(pipeline,
                                param_grid,
                                cv=3,
                                scoring='r2',
                                n_jobs=n_jobs,
                                )
    
    # Fit the pipeline
    grid_search.fit(X, y)

    #predict (train)
    y_pred = grid_search.predict(X)

    #performance (train)
    fullmodel = grid_search
    model = grid_search.best_estimator_

    #predict (test)
    y_pred_test = model.predict(X_test)

    return fullmodel, model, y_pred, y_pred_test

In [5]:
def MLsvr(X, y, X_test):
    # Define the pipeline
    pipeline = Pipeline([('scaler', StandardScaler()),  # Standardize features
                         ('model', TransformedTargetRegressor(
                             regressor=SVR(),  #  the model
                             transformer=Pipeline([
                                 ('scaler', StandardScaler()),
                                 ('power', PowerTransformer(method='yeo-johnson')),  # Apply PowerTransformer to the target
                                 ])
                          ))
                        ])
    
    # Define hyperparameter grid for ml
    param_grid = {
                    'model__regressor__C': np.logspace(-2, 2, 50),
                    'model__regressor__gamma': np.logspace(-3, 1, 50),
                    'model__regressor__kernel': ['rbf', 'linear']
                }
    
    # Set up GridSearchCV
    grid_search = GridSearchCV(pipeline,
                                param_grid,
                                cv=3,
                                scoring='r2',
                                n_jobs=n_jobs,
                                )
    
    # Fit the pipeline
    grid_search.fit(X, y)

    #predict (train)
    y_pred = grid_search.predict(X)

    #performance (train)
    fullmodel = grid_search
    model = grid_search.best_estimator_

    #predict (test)
    y_pred_test = model.predict(X_test)

    return fullmodel, model, y_pred, y_pred_test

In [6]:
def MLrf(X, y, X_test):
    # Define the pipeline
    pipeline = Pipeline([('scaler', StandardScaler()),  # Standardize features
                         ('model', TransformedTargetRegressor(
                             regressor=RandomForestRegressor(random_state=42),  #  the model
                             transformer=Pipeline([
                                 ('scaler', StandardScaler()),
                                 ('power', PowerTransformer(method='yeo-johnson')),  # Apply PowerTransformer to the target
                                 ])
                          ))
                        ])
    
    # Define hyperparameter grid for ml
    param_grid = {
                    'model__regressor__n_estimators': [100, 200, 300, 500,5000],               
                    'model__regressor__max_depth': [None, 10, 20, 30, 50],                
                    'model__regressor__min_samples_split': [2, 5, 10],                    
                    'model__regressor__min_samples_leaf': [1, 2, 4],                      
                    'model__regressor__max_features': ['sqrt', 'log2'],          
                }
    
    # Set up GridSearchCV
    grid_search = GridSearchCV(pipeline,
                                param_grid,
                                cv=3,
                                scoring='r2',
                                n_jobs=n_jobs,
                                )
    
    # Fit the pipeline
    grid_search.fit(X, y)

    #predict (train)
    y_pred = grid_search.predict(X)

    #performance (train)
    fullmodel = grid_search
    model = grid_search.best_estimator_

    #predict (test)
    y_pred_test = model.predict(X_test)

    return fullmodel, model, y_pred, y_pred_test

In [7]:
def MLxgb(X, y, X_test):
    # Define the pipeline
    pipeline = Pipeline([('scaler', StandardScaler()),  # Standardize features
                         ('model', TransformedTargetRegressor(
                             regressor=XGBRegressor(objective='reg:squarederror', random_state=42),  #  the model
                             transformer=Pipeline([
                                 ('scaler', StandardScaler()),
                                 ('power', PowerTransformer(method='yeo-johnson')),  # Apply PowerTransformer to the target
                                 ])
                          ))
                        ])
    
    # Define hyperparameter grid for ml
    param_grid = {
                    'model__regressor__n_estimators': [100, 200, 300, 500],
                    'model__regressor__max_depth': [1, 3, 5, 7, 9],
                    'model__regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
                    'model__regressor__gamma': [0, 1, 5], 
                    'model__regressor__subsample': [0.5, 0.7, 1.0]
                }
    
    # Set up GridSearchCV
    grid_search = GridSearchCV(pipeline,
                                param_grid,
                                cv=3,
                                scoring='r2',
                                n_jobs=n_jobs,
                                )
    
    # Fit the pipeline
    grid_search.fit(X, y)

    #predict (train)
    y_pred = grid_search.predict(X)

    #performance (train)
    fullmodel = grid_search
    model = grid_search.best_estimator_

    #predict (test)
    y_pred_test = model.predict(X_test)

    return fullmodel, model, y_pred, y_pred_test

In [8]:
#path to dir
path='/scratch2/alinat/project/PD-EEG-ANON_eegOnly/MLtables180'

In [9]:
#save/load prepared data
fnm = '/MLout/DCT_1targs_anat_rest_4fold_ZnoLG.pkl'
# # Save to file
# with open(path+fnm, 'wb') as file:
#     pickle.dump(Folds, file)

# Load from file
with open(path+fnm, 'rb') as file:
    Folds = pickle.load(file)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
# Ignore FutureWarnings
simplefilter(action="ignore", category=FutureWarning)
# Ignore ConvergenceWarnings (common in ML models)
simplefilter(action="ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

#ML level 1

MLlvl1 = {}

for fold in sorted(Folds.keys()): #folds
    print(fold, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    MLlvl1[fold] = {}

    for mlfunc, mlname in zip([MLenet, MLpls, MLkrr, MLsvr, MLrf, MLxgb],
                              ['enet', 'pls', 'krr', 'svr', 'rf', 'xgb']): #ML models
        print(mlname, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
        MLlvl1[fold][mlname] = {}

        for trg in Folds[fold]['Train']['target']: 
            print(trg, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            
            MLlvl1[fold][mlname][trg] = {}
            MLlvl1[fold][mlname][trg]['fullmodels'] = {}
            MLlvl1[fold][mlname][trg]['models'] = {}

            #training set
            targ = Folds[fold]['Train']['target'][trg]
            feat = Folds[fold]['Train']['features']
            #testing set
            targ_test = Folds[fold]['Test']['target'][trg]
            feat_test = Folds[fold]['Test']['features']
            
            #for storing
            dct_train = {}
            dct_test = {}
            
            for key in sorted(feat.keys()): #features
                print(key, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                #target table
                y = targ
                y_test = targ_test
                #feature table
                X = feat[key]
                X_test = feat_test[key]
                
                dct_train['y_obs'] = y
                dct_test['y_obs'] = y_test

                #ml function
                fullmodel, model, y_pred, y_pred_test = mlfunc(X, y, X_test)
                
                #save some train performance
                MLlvl1[fold][mlname][trg]['fullmodels']['fullmodel__'+key] = fullmodel
                MLlvl1[fold][mlname][trg]['models']['model__'+key] = model
                dct_train[key] = y_pred
                dct_test[key] = y_pred_test
            
            MLlvl1[fold][mlname][trg]['predval_train'] = pd.DataFrame(dct_train)
            MLlvl1[fold][mlname][trg]['predval_test'] = pd.DataFrame(dct_test)
    print('Done', fold, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


        

In [11]:
#save
fnm = '/MLout/MLlvl1_1targ_6algs_ZnoLG.pkl'
# # Save to file
with open(path+fnm, 'wb') as file:
     pickle.dump(MLlvl1, file)

# Load from file
# with open(path+fnm, 'rb') as file:
#     Folds = pickle.load(file)